# 04. 피해 규모 예측 모델 (12번 ①)

**질문:** 날씨 정보만 보고, 비 온 날과 그 뒤 며칠 동안 외지인 결제건수가 얼마나 줄어들지 미리 맞힐 수 있나?

- 대상: 춘천 외지인(83개 업종 합계 + 업종 7그룹), 강남 외지인(83개 업종 합계)
- 기간: 2025-07-01 ~ 2025-12-31 (184일), 표본 날은 주 모형(`event_study.py`)과 같음
- 코드: `src/model.py` (이 노트북은 그 함수를 불러서 실행하고 결과를 보여 줍니다)
- 상태: **팀 검증 전**

## 이 셀이 하는 일: 실행 위치와 환경 확인
- 노트북이 `notebooks/` 안에서 실행되면 프로젝트 폴더로 한 칸 올라갑니다 (코드가 상대경로를 쓰기 때문).
- 결과가 라이브러리 버전에 따라 조금 달라질 수 있어서, 버전을 먼저 찍어 둡니다.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':   # notebooks 폴더에서 실행된 경우
    os.chdir('..')                                   # 프로젝트 폴더로 이동
sys.path.insert(0, 'src')

import numpy, pandas, statsmodels, sklearn
print('실행 폴더:', os.path.basename(os.getcwd()))
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| statsmodels', statsmodels.__version__, '| scikit-learn', sklearn.__version__)

실행 폴더: bigcontest_team
Python 3.14.7 | numpy 2.5.2 | pandas 3.0.5 | statsmodels 0.15.0 | scikit-learn 1.9.1


## 평가 방법 (쉬운 설명)

**사건 하나씩 빼고 검증 (leave-one-event-out)**
1. 호우 사건(30mm 이상 비가 이어진 날 묶음) 하나를 고릅니다.
2. 그 사건의 [시작 전날 ~ 끝+7일]을 학습 자료에서 뺍니다. 모델은 이 비를 한 번도 못 본 상태가 됩니다.
3. 나머지 날로 학습한 뒤, 뺀 사건의 [시작일 ~ 끝+5일]을 예측합니다.
4. 사건마다 반복합니다 (춘천 12개, 서울은 평가 가능한 9개).

**비교하는 모델 3개**
| 이름 | 쓰는 정보 | 뜻 |
|---|---|---|
| base | 요일·월·공휴일 | 기준 모델: "날씨를 모르면 이 정도 맞힌다" |
| ols_wx | base + 날씨 | 회귀(선을 긋는 모델) |
| gbm_wx | base + 날씨 | 트리 모델(조건을 여러 번 나눠 맞히는 모델) |

날씨 변수: 일강수량, 30mm 이상 여부, 10~30mm 여부, 최고기온, 33℃ 이상 여부, 마지막 호우일로부터 1~5일째 여부.
결제건수의 과거값은 넣지 않습니다 → 미래 정보가 새어 들어가지 않습니다(데이터 누수 없음).

**지표**
- `MAE`: 예측과 실제의 차이(로그 기준) 평균. 0.01은 대략 1% 차이. **작을수록 좋음.**
- `개선률%`: 기준 모델보다 오차가 몇 % 줄었나. **양수면 날씨가 예측에 도움이 됨.**
- `base보다_나은_사건`: 사건 중 몇 개에서 기준 모델보다 오차가 작았나.
- `실제손실%`: 호우일에 "평소(기준 모델 예측)"보다 실제로 줄어든 비율.
- `손실오차%p`: 모델이 예측한 호우일 손실과 실제 손실의 차이(%p) 평균.

## 이 셀이 하는 일: 데이터 읽기 (카드·기상·호우 사건·달력, 약 30초)

In [2]:
import pandas as pd
from event_study import load_all
from model import run_all

data = load_all()
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
pd.set_option('display.width', 200)

## 이 셀이 하는 일: 모든 대상에 대해 사건 하나씩 빼고 검증 실행

In [3]:
detail, summary = run_all(data)

## 결과 1. 오차 비교 (작을수록 좋음)

In [4]:
summary[['지역', '대상', '평가 사건 수', 'MAE_base', 'MAE_ols_wx', 'MAE_gbm_wx']]

,지역,대상,평가 사건 수,MAE_base,MAE_ols_wx,MAE_gbm_wx
0,춘천,83합계,12,0.097,0.077,0.071
1,춘천,식료품,12,0.092,0.078,0.075
2,춘천,외식,12,0.105,0.088,0.074
3,춘천,의료·약국,12,0.088,0.084,0.090
4,춘천,교통·숙박,12,0.129,0.098,0.099
5,춘천,여가,12,0.181,0.185,0.146
6,춘천,상품쇼핑,12,0.182,0.158,0.133
7,춘천,생활·교육서비스,12,0.117,0.120,0.111
8,강남,83합계,9,0.034,0.039,0.034


## 결과 2. 날씨가 도움이 되었나 (기준 모델 대비)

In [5]:
summary[['지역', '대상', '개선률%_ols_wx', 'base보다_나은_사건_ols_wx',
         '개선률%_gbm_wx', 'base보다_나은_사건_gbm_wx']]

,지역,대상,개선률%_ols_wx,base보다_나은_사건_ols_wx,개선률%_gbm_wx,base보다_나은_사건_gbm_wx
0,춘천,83합계,20.684,9/12,27.316,11/12
1,춘천,식료품,16.056,7/12,18.682,8/12
2,춘천,외식,15.910,10/12,29.568,9/12
3,춘천,의료·약국,5.005,5/12,-2.546,4/12
4,춘천,교통·숙박,24.021,6/12,23.121,8/12
5,춘천,여가,-2.215,6/12,18.966,11/12
6,춘천,상품쇼핑,13.550,7/12,26.872,10/12
7,춘천,생활·교육서비스,-2.939,4/12,5.021,8/12
8,강남,83합계,-15.264,2/9,0.091,4/9


## 결과 3. 호우일 손실을 얼마나 맞혔나 (%p, 작을수록 좋음)

In [6]:
summary[['지역', '대상', '실제손실%_평균', '손실오차%p_ols_wx', '손실오차%p_gbm_wx']]

,지역,대상,실제손실%_평균,손실오차%p_ols_wx,손실오차%p_gbm_wx
0,춘천,83합계,15.051,7.832,8.042
1,춘천,식료품,12.711,8.240,9.362
2,춘천,외식,15.131,8.985,8.775
3,춘천,의료·약국,6.541,7.262,8.825
4,춘천,교통·숙박,17.205,10.115,11.131
5,춘천,여가,26.269,14.540,14.175
6,춘천,상품쇼핑,18.571,15.052,11.381
7,춘천,생활·교육서비스,2.643,10.365,9.681
8,강남,83합계,2.692,4.422,3.321


## 결과 4. 사건별 결과: 춘천 외지인 83합계

In [7]:
detail[('춘천', '83합계')]

,사건,시험일수,학습일수,MAE_base,MAE_ols_wx,MAE_gbm_wx,실제손실%,예측손실%_ols_wx,예측손실%_gbm_wx
0,07/15~07/17,8,152,0.191,0.086,0.168,24.687,26.584,4.713
1,07/20,6,154,0.042,0.073,0.092,1.254,26.321,17.667
2,08/06,6,155,0.061,0.069,0.044,21.227,18.672,18.221
3,08/26,6,154,0.092,0.041,0.061,19.754,17.742,22.514
4,09/01,6,154,0.083,0.075,0.074,5.717,11.933,15.962
5,09/04,6,154,0.076,0.064,0.039,2.623,10.321,9.500
6,09/06,6,154,0.085,0.067,0.033,20.492,6.926,12.419
7,09/13,6,154,0.067,0.037,0.038,13.265,13.441,20.085
8,09/17,6,154,0.074,0.075,0.044,12.798,15.226,14.428
9,09/20,6,154,0.063,0.054,0.042,6.197,14.277,12.294


## 결과 5. 사건별 결과: 강남 외지인 83합계

In [8]:
detail[('강남', '83합계')]

,사건,시험일수,학습일수,MAE_base,MAE_ols_wx,MAE_gbm_wx,실제손실%,예측손실%_ols_wx,예측손실%_gbm_wx
0,07/16~07/17,7,149,0.051,0.041,0.042,9.306,4.118,1.560
1,07/19,6,150,0.020,0.026,0.016,5.324,8.403,4.234
2,08/29,6,150,0.015,0.028,0.026,2.231,-1.143,3.043
3,09/04,6,150,0.049,0.062,0.046,6.839,0.912,3.898
4,09/06,6,150,0.036,0.043,0.036,5.175,0.613,3.643
5,09/13,6,150,0.017,0.025,0.020,-0.158,5.800,4.961
6,09/17,6,150,0.025,0.035,0.030,1.336,4.736,3.947
7,09/20,6,150,0.044,0.047,0.045,-4.544,1.677,2.164
8,09/24,4,154,0.049,0.046,0.043,-1.278,0.813,0.054


## 읽을 때 주의할 점
- **사건 수가 적습니다** (춘천 12개, 강남 9개). 사건 하나가 평균을 크게 움직이므로, 표의 숫자는 "경향"으로만 읽습니다.
- **서울 10/10 사건은 평가에서 빠집니다.** 그날이 앞 10/06 사건(연휴로 제외된 사건)의 구간 [10/06~10/13] 안에 있어 표본 밖이기 때문입니다. 주 모형에서도 이 사건의 호우일은 추정에 쓰이지 않고, 10/14~10/17(사건 뒤 4~7일)만 표본에 들어갑니다.
- **"실제손실"의 기준인 base 모델은 다른 비 온 날도 학습에 포함합니다.** 그래서 평소 수준을 조금 낮게 잡을 수 있고, 실제손실이 주 모형의 L(춘천 21.5%)보다 작게 나옵니다.
- 2025년 하반기는 호우가 유난히 잦은 시즌입니다. 춘천은 14개로 1995~2024년 어느 시즌(최대 13개)보다 많고, 서울은 13개로 30년 최대와 같습니다. 그래서 평범한 해에도 같은 성능이 나올지는 알 수 없습니다.
- 상관관계를 보는 모델입니다. "비 때문에 줄었다"는 인과로 쓰지 않고 "관련이 있다"로 씁니다.